# John Snow's Cholera Outbreak Analysis (1854)




In [ ]:
## Libraries & Data Loading

import geopandas as gpd
import folium
from folium.plugins import HeatMap

# Load shapefiles and convert to WGS84 for web mapping
deaths = gpd.read_file('SnowGIS_SHP/SnowGIS_SHP/Cholera_Deaths.shp')
pumps = gpd.read_file('SnowGIS_SHP/SnowGIS_SHP/Pumps.shp')

deaths = deaths.to_crs(epsg=4326)
pumps = pumps.to_crs(epsg=4326)

print(f"Number of death locations {len(deaths)} ")
print(f"Total deaths {deaths['Count'].sum()}")
print(f"Number of water pumps {len(pumps)}")


Number of death locations 250 
Total deaths 489
Number of water pumps 8


## Task#1 : Replicating John Snow's visualization



In [ ]:
# Calculate map center
center_lat = deaths.geometry.y.mean()
center_lon = deaths.geometry.x.mean()

# Create map
m1 = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles='OpenStreetMap')

# then we add deaths as circles (size proportional to count)
for idx, row in deaths.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=row['Count'] * 1.5,
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.5,
        popup=f"Deaths: {row['Count']}"
    ).add_to(m1)

# we add pumps as blue markers
for idx, row in pumps.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        color='blue',
        fill=True,
        fillColor='blue',
        popup='Water Pump'
    ).add_to(m1)

m1.save('cholera_proportional_map.html')
m1


## Task#2: Identifying the outbreak's center 


In [ ]:
# Create new map
m2 = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles='OpenStreetMap')

# repeat coordinates by death count for intensity
heat_data = []
for idx, row in deaths.iterrows():
    for _ in range(int(row['Count'])):
        heat_data.append([row.geometry.y, row.geometry.x])

# adding a heatmap
HeatMap(heat_data, radius=15, blur=20).add_to(m2)

# then we add pump markers
for idx, row in pumps.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color='blue',
        fill=True,
        popup='Pump'
    ).add_to(m2)


m2.save('cholera_heatmap.html')
m2


## Conclusion

The heatmap clearly shows the concentration of deaths around one central pump (Broad Street pump), supporting John Snow's hypothesis that cholera was because of water and not airborne.
